# Multi-Modal LLM using EdenAI for image reasoning

In this notebook, we show how to use MultiModal LLM class for image understanding/reasoning.


In [ ]:
%pip install llama-index-multi-modal-llms-edenai

## Load and initialize EdenAI 

In [ ]:
import os


EDENAI_API_KEY = ""
os.environ["EDENAI_API_KEY"] = EDENAI_API_KEY


## Download Images and Load Images locally

In [ ]:

import os
import asyncio
import pandas as pd
from PIL import Image, UnidentifiedImageError
import requests
from io import BytesIO

from llama_index.core.schema import ImageDocument
from llama_index.core.llms import ChatMessage, MessageRole

from llama_index.multi_modal_llms.edenai import EdenaiMultiModal

image_sources = [
    "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRFU7U2h0umyF0P6E_yhTX45sGgPEQAbGaJ4g&s",
    "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/481px-Cat03.jpg",
]
image_documents = [
    ImageDocument(image_url=url) for url in image_sources
]

if not image_documents:
    raise RuntimeError("Could not create any ImageDocument objects from URLs.")

prompts = [
    "Describe this image in detail.",
    "What animal is shown here?",
    "What is the main subject of the picture?",
]

llm_model = "openai/gpt-4o" # or another model Eden AI supports
try:
    multi_modal_llm = EdenaiMultiModal(
        model=llm_model,
        api_key=EDENAI_API_KEY,
    )
    print(f"Initialized EdenaiMultiModal with model: {llm_model}")
except Exception as e:
    print(f"FATAL: Failed to initialize EdenaiMultiModal: {e}")
    raise e

In [ ]:
print("\n--- Testing complete (Sync) ---")
complete_results = []
for prompt in prompts:
    image_doc_to_test = image_documents[0]
    print(f"\nTesting prompt: '{prompt}' with image URL: {image_doc_to_test.image_url}") 
    try:
        response = multi_modal_llm.complete(
            prompt=prompt,
            image_documents=[image_doc_to_test],
        )
        print(f"Response Text: {response.text}")
        print(f"Token Usage: {response.additional_kwargs}")
        complete_results.append({"prompt": prompt, "image_url": image_doc_to_test.image_url, "response_text": response.text})
        assert isinstance(response.text, str) and len(response.text) > 0
    except Exception as e:
        print(f"Error during complete: {e}")
        complete_results.append({"prompt": prompt, "image_url": image_doc_to_test.image_url, "response_text": f"ERROR: {e}"})

In [ ]:
print("\n--- Testing acomplete (Async) ---")
acomplete_results = []

async def run_acomplete():
    for prompt in prompts:
        if len(image_documents) > 1:
            image_doc_to_test = image_documents[1]
        else:
            image_doc_to_test = image_documents[0]
        print(f"\nTesting prompt: '{prompt}' with image URL: {image_doc_to_test.image_url}") 
        try:
            response = await multi_modal_llm.acomplete(
                prompt=prompt,
                image_documents=[image_doc_to_test],
            )
            print(f"Response Text: {response.text}")
            print(f"Token Usage: {response.additional_kwargs}")
            acomplete_results.append({"prompt": prompt, "image_url": image_doc_to_test.image_url, "response_text": response.text}) 
            assert isinstance(response.text, str) and len(response.text) > 0
        except Exception as e:
            print(f"Error during acomplete: {e}")
            acomplete_results.append({"prompt": prompt, "image_url": image_doc_to_test.image_url, "response_text": f"ERROR: {e}"}) 

try:
    import nest_asyncio
    nest_asyncio.apply()
    asyncio.run(run_acomplete())
except RuntimeError as e:
    print(f"Error during async execution: {e}")
    raise e


In [ ]:
print("\n--- Testing chat (Sync) ---")
chat_results = []
for prompt in prompts:
    image_doc_to_test = image_documents[0]
    print(f"\nTesting prompt: '{prompt}' with image URL: {image_doc_to_test.image_url}")

    messages = [
        ChatMessage(
            role=MessageRole.USER,
            content=prompt,
            additional_kwargs={"image_documents": [image_doc_to_test]}
        )
    ]

    try:
        response = multi_modal_llm.chat(messages=messages)
        print(f"Response Content: {response.message.content}")
        print(f"Token Usage: {response.additional_kwargs}")
        chat_results.append({"prompt": prompt, "image_url": image_doc_to_test.image_url, "response_content": response.message.content}) 
        assert response.message.role == MessageRole.ASSISTANT
        assert isinstance(response.message.content, str) and len(response.message.content) > 0
    except Exception as e:
        print(f"Error during chat: {e}")
        chat_results.append({"prompt": prompt, "image_url": image_doc_to_test.image_url, "response_content": f"ERROR: {e}"}) 

In [ ]:
print("\n--- Testing achat (Async) ---")
achat_results = []

async def run_achat():
    for prompt in prompts:
        if len(image_documents) > 1:
            image_doc_to_test = image_documents[1]
        else:
            image_doc_to_test = image_documents[0]
        print(f"\nTesting prompt: '{prompt}' with image URL: {image_doc_to_test.image_url}") 

        messages = [
            ChatMessage(
                role=MessageRole.USER,
                content=prompt,
                additional_kwargs={"image_documents": [image_doc_to_test]}
            )
        ]

        try:
            response = await multi_modal_llm.achat(messages=messages)
            print(f"Response Content: {response.message.content}")
            print(f"Token Usage: {response.additional_kwargs}")
            achat_results.append({"prompt": prompt, "image_url": image_doc_to_test.image_url, "response_content": response.message.content}) 
            assert response.message.role == MessageRole.ASSISTANT
            assert isinstance(response.message.content, str) and len(response.message.content) > 0
        except Exception as e:
            print(f"Error during achat: {e}")
            achat_results.append({"prompt": prompt, "image_url": image_doc_to_test.image_url, "response_content": f"ERROR: {e}"}) 

try:
    asyncio.run(run_achat())
except RuntimeError as e:
     if "cannot run current event loop" in str(e) and 'JPY_PARENT_PID' in os.environ:
         print("Asyncio loop already running (likely in Jupyter). Trying nest_asyncio.")
         import nest_asyncio
         nest_asyncio.apply()
         asyncio.run(run_achat())
     else:
         raise e


In [ ]:
print("\n--- Testing stream_completion (Sync) ---")

full_response_text = ""
try:
    stream = multi_modal_llm.stream_complete(
        prompt=prompt,
        image_documents=[image_doc_to_test],
    )
    print("Stream Chunks:")
    for chunk in stream:
        if chunk.delta:
            print(chunk.delta, end="", flush=True)
            full_response_text += chunk.delta
    print("\n--- End of Stream ---")
    print(f"Aggregated Response: {full_response_text}")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
print("\n--- Testing stream_chat (Sync) ---")
stream_chat_results = []
test_prompt = "Is the animal in the image looking at the camera?"
image_doc_to_test = image_documents[0]
print(f"\nTesting stream chat prompt: '{test_prompt}' with image URL: {image_doc_to_test.image_url}") 

messages = [
    ChatMessage(
        role=MessageRole.USER,
        content=test_prompt,
        additional_kwargs={"image_documents": [image_doc_to_test]}
    )
]

full_response_text = ""
try:
    stream = multi_modal_llm.stream_chat(messages=messages)
    print("Stream Chunks:")
    chunk_count = 0
    for chunk in stream:
        chunk_count += 1
        delta = chunk.delta or ""
        print(delta, end="", flush=True)
        full_response_text += delta
    print("\n--- End of Stream ---")
    print(f"\nAggregated Response: {full_response_text}")
    stream_chat_results.append({"prompt": test_prompt, "image_url": image_doc_to_test.image_url, "full_response": full_response_text})
    assert chunk_count > 0
    assert len(full_response_text) > 0
except Exception as e:
    print(f"\nError during stream_chat: {e}")
    stream_chat_results.append({"prompt": test_prompt, "image_url": image_doc_to_test.image_url, "full_response": f"ERROR: {e}"})